# Phân tích đa chuỗi thời gian theo vùng

Mỗi chuỗi gồm số đơn tiếp nhận theo ngày của một vùng trong LaDe-D Jilin. Phân tích dùng toàn bộ giai đoạn quan sát, còn cửa sổ 27/06–02/10 dùng để đối chiếu độ nhạy.

Trong ngày thành phố có dữ liệu, vùng không có đơn ghi nhận được gán 0. Ngày cả thành phố thiếu bản ghi được để trống. Năm kỹ thuật trên trục ngày là 2022.

In [ ]:
from pathlib import Path
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.config import DEMAND_DATE_COLUMN as DATE
from src.data_loader import loadRawDelivery
from src.regional_time_series import STABLE_PERIOD, buildRegionalPanel, compareRegions, summarizeRegions

panel = buildRegionalPanel(loadRawDelivery())
daily = panel.pivot(index=DATE, columns="region_id", values="demand").astype("float64")
display(summarizeRegions(panel).round(2))
unobserved = daily.index[daily.isna().all(axis=1)]
print("Ngày thiếu quan sát:", ", ".join(unobserved.strftime("%d/%m")))

## Số đơn tiếp nhận theo ngày

Đường biểu diễn ngắt ở những ngày thiếu quan sát. Giá trị 0 chỉ dùng cho ngày có dữ liệu thành phố nhưng vùng không ghi nhận đơn.

In [ ]:
ax = daily.plot(figsize=(12, 5), linewidth=1.3, ylabel="Số đơn", xlabel="Ngày (ngày/tháng)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.legend(title="Mã vùng", ncol=4)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Tương quan giữa các vùng

So sánh Pearson trên mức số đơn cùng ngày và trên sai phân của hai ngày lịch liền nhau có dữ liệu. Bảng ghi số ngày ghép cặp cho từng hệ số.

In [ ]:
full = compareRegions(panel)
stable = compareRegions(panel.loc[panel[DATE].between(*STABLE_PERIOD)])
display(full.round(3))
display(stable.round(3))

labels = {"level": "Số đơn cùng ngày", "daily_change": "Thay đổi so với ngày liền trước"}
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, (transform, title) in zip(axes, labels.items()):
    pairs = full.loc[full["transform"].eq(transform), ["region_a", "region_b", "pearson_r"]]
    reverse = pairs.rename(columns={"region_a": "region_b", "region_b": "region_a"})
    matrix = pd.concat([pairs, reverse]).pivot(index="region_a", columns="region_b", values="pearson_r")
    heatmap = ax.imshow(matrix, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(matrix.columns)), labels=matrix.columns)
    ax.set_yticks(range(len(matrix.index)), labels=matrix.index)
    ax.set_title(title)
    for row, values in enumerate(matrix.to_numpy()):
        for col, value in enumerate(values):
            if pd.notna(value):
                ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                        color="white" if abs(value) > 0.65 else "black")
fig.colorbar(heatmap, ax=axes, label="Pearson r")
plt.show()

## Diễn giải

Toàn kỳ có 163 ngày ghép cặp theo mức số đơn và 158 ngày ghép cặp theo sai phân. Vùng 53 và 129 có tương quan khoảng 0,81 theo mức và 0,51 theo sai phân. Vùng 31 và 165 có tương quan khoảng 0,53 theo mức và gần 0 theo sai phân.

Thứ hạng thay đổi khi chỉ xét 27/06–02/10. Chẳng hạn, tương quan theo mức của vùng 129 và 165 tăng từ khoảng 0,49 lên 0,60. Vì vậy, báo cáo diễn giải từng hệ số theo cách đo và giai đoạn quan sát tương ứng.